# gCastle API Overview

This notebook explores the core components and API of gCastle, a causal structure learning library by Huawei Noah's Ark Lab.

**What you'll learn:**
- How to generate synthetic causal data
- How to run causal discovery algorithms
- How to evaluate causal discovery results
- How to visualize causal graphs

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import matplotlib.pyplot as plt

import tutorials.gCastle.gCastle_utils as tgcutil

logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

plt.rcParams['figure.figsize'] = (10, 6)

## Data Generation

Generate synthetic data from a linear Gaussian causal model with a random DAG.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data and ground truth DAG
num_samples = 500
num_vars = 5
edge_density = 0.4

data, true_dag = tgcutil.generate_linear_gaussian_data(
    num_samples=num_samples,
    num_vars=num_vars,
    edge_density=edge_density,
    random_state=42
)

print(f"Data shape: {data.shape}")
print(f"Number of edges in true DAG: {len(true_dag.edges())}")
print(f"Edges: {list(true_dag.edges())}")

### Visualize True Causal Graph

In [ ]:
true_adj = tgcutil.dag_to_adjacency(true_dag, num_vars)
tgcutil.plot_dag(true_adj, title="True Causal Graph")
plt.show()

## Data Normalization

Most causal discovery algorithms benefit from normalized data.

In [ ]:
# Normalize data to zero mean and unit variance
normalized_data = tgcutil.normalize_data(data)

print(f"Data mean: {normalized_data.mean(axis=0).round(4)}")
print(f"Data std: {normalized_data.std(axis=0).round(4)}")

## Evaluation Metrics

Key metrics for evaluating causal discovery:
- **FDR** (False Discovery Rate): Proportion of incorrectly discovered edges
- **TPR** (True Positive Rate): Proportion of correctly discovered edges
- **FPR** (False Positive Rate): Proportion of incorrect edges among non-edges
- **SHD** (Structural Hamming Distance): Total number of differences

In [ ]:
# Compute metrics for a dummy estimated adjacency matrix
estimated_adj = np.zeros((num_vars, num_vars))
# Add one correct edge
if true_adj[0, 1] == 1:
    estimated_adj[0, 1] = 1
# Add one incorrect edge
estimated_adj[1, 2] = 1

metrics = tgcutil.compute_dag_metrics(estimated_adj, true_adj)

print("Metrics:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

## Causal Discovery Algorithms

gCastle provides various algorithms for causal structure learning:
- **LiNGAM**: Linear Non-Gaussian Acyclic Model
- **GES**: Greedy Equivalence Search
- **PC**: PC algorithm (constraint-based)
- **NOTEARS**: Neural Ordered Transforms for Acyclic Relationships

In [ ]:
from gcastle.algorithms import LinearGES, PC, NOTEARS

# Define algorithms to compare
algorithms = {
    "LinearGES": (LinearGES, {}),
    "PC": (PC, {"alpha": 0.05}),
}

### Run and Compare Algorithms

In [ ]:
# Run algorithms and compare
_LOG.info("Running causal discovery algorithms...")
results_df = tgcutil.compare_algorithms(
    normalized_data,
    true_adj,
    algorithms
)

print("\nComparison Results:")
print(results_df.to_string(index=False))

### Visualize Results

In [ ]:
tgcutil.plot_comparison_metrics(
    results_df,
    metrics_to_plot=["fdr", "tpr", "shd"]
)
plt.show()